In [1]:
# imports
import json
import re
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List
from datetime import datetime
from IPython.display import display, HTML
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

try:
    import requests
except ImportError:
    requests = None

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate" 
OLLAMA_MODEL = "llama3" 

 ### מבנה תוצאה אחיד לסוכנים

במערכת קיימים מספר סוכנים המבצעים משימות שונות.
כדי לאפשר ל-OrchestratorAgent לעבוד מול כל הסוכנים בצורה אחידה,
כל סוכן מחזיר אובייקט מסוג AgentResult המכיל את תוצאת הפעולה,
רמת הביטחון ומידע נוסף במידת הצורך.

In [3]:
# AgentResult
@dataclass
class AgentResult:
    agent_name: str     # שם הסוכן שהחזיר את התוצאה
    output: Any          # תוצאת הפעולה
    confidence: float = 1.0                      # רמת ביטחון בין 0 ל-1
    metadata: Optional[Dict[str, Any]] = None  # מידע נוסף (אופציונלי)

### AgentResult דוגמה לשימוש ב- 

הדוגמה הבאה מציגה כיצד סוכן מחזיר תוצאה בפורמט האחיד של המערכת.
במקרה זה RouterAgent זיהה שהמשתמש מבקש שיחה כללית (generalChat)
והחזיר רמת ביטחון של 0.82.

In [4]:
result = AgentResult("RouterAgent", {"intent": "generalChat"}, 0.82)
print(result.agent_name)   # "RouterAgent"
print(result.confidence)   # 0.82

RouterAgent
0.82


### הצגת הודעות בממשק

כדי לשפר את הקריאות של תוצאות הריצה במחברת,
נוצרה פונקציית עזר המציגה הודעות בפורמט מעוצב.
הודעות הצלחה, שגיאה, אזהרה ומידע מוצגות בצבעים ובאייקונים שונים,
וכך קל יותר לעקוב אחר פעולת הסוכנים והמערכת.

In [5]:
def print_result(message: str, status: str = "success"):
    colors = {
        "success": "#2ecc71",   # ירוק
        "error":   "#e74c3c",   # אדום
        "warning": "#f39c12",   # כתום
        "info":    "#3498db"    # כחול
    }
    icons = {
        "success": "✅",
        "error":   "❌",
        "warning": "⚠️",
        "info":    "ℹ️"
    }
    color = colors.get(status, "#3498db")
    icon  = icons.get(status, "ℹ️")
    
    html = f"""
    <div style="
        background-color: {color}22;
        border-left: 4px solid {color};
        padding: 10px 15px;
        margin: 5px 0;
        border-radius: 4px;
        font-family: Arial;
        font-size: 14px;
        color: #2c3e50;
    ">
        {icon} {message}
    </div>
    """
    display(HTML(html))

### ישויות מערכת התשלומים

מערכת התשלומים מבוססת על ארבע ישויות מרכזיות:
משתמש (User), ארנק (Wallet), עסקה (Transaction) ובקשת תשלום (PaymentRequest).

כל ישות מיוצגת באמצעות Data Class, המאפשרת לשמור את המידע בצורה מסודרת וברורה.
ישויות אלו מהוות את בסיס הנתונים הלוגי של המערכת וכל הסוכנים עובדים מולן.

In [6]:
# ישויות הנתונים המרכזיות של מערכת התשלומים
# מחלקות אלו מייצגות משתמשים, ארנקים, עסקאות ובקשות תשלום
# ומשמשות לאחסון המידע העסקי של המערכת

@dataclass
class User:
    user_id: str            
    name: str            # שם המשתמש
    phone_number: str    # מספר טלפון

@dataclass
class Wallet:
    user_id: str      
    balance: float       # יתרה נוכחית בארנק

@dataclass
class Transaction:
    sender_id: str       # שולח הכסף
    receiver_id: str     # מקבל הכסף
    amount: float        # סכום ההעברה
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())  # זמן ביצוע העסקה
    status: str = "completed"   # סטטוס העסקה
    risk_score: float = 0.0     # ציון סיכון לזיהוי הונאות

@dataclass
class PaymentRequest:
    request_id: str      # מזהה בקשת התשלום
    requester_id: str    # המשתמש שביקש את התשלום
    payer_id: str        # המשתמש שאמור לשלם
    amount: float        # סכום הבקשה
    status: str = "pending"     # סטטוס הבקשה
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())  # זמן יצירה

### אחסון נתוני המערכת

בפרויקט זה הנתונים נשמרים בזיכרון  
באמצעות מילונים ורשימות .

גישה זו מאפשרת לממש את הלוגיקה העסקית של מערכת התשלומים
ללא צורך במסד נתונים חיצוני, תוך שמירה על פשטות המערכת
והתמקדות בארכיטקטורת הסוכנים.

In [7]:
# מסד הנתונים של המערכת - מילונים פשוטים בזיכרון
users_db: Dict[str, User] = {}
wallets_db: Dict[str, Wallet] = {}
transactions_db: List[Transaction] = []
payment_requests_db: Dict[str, PaymentRequest] = {}
audit_log: List[str] = []
request_counter = [0]  # רשימה כדי לאפשר שינוי בתוך פונקציה

### הגדרת הכוונות במערכת

במערכת מרובת סוכנים, כל בקשת משתמש מסווגת תחילה לכוונה מסוימת.
הכוונה מייצגת את הפעולה שהמשתמש מעוניין לבצע.

לאחר זיהוי הכוונה, הבקשה מועברת לרכיב התזמור,
אשר מחליט איזה סוכן או כלי צריך לבצע את הפעולה.

In [8]:
INTENTS = [
    "createUser", "checkBalance", "transferMoney",
    "requestPayment", "approvePayment", "rejectPayment",
    "showTransactions", "fraudCheck", "securityReview",
    "explainLastAction", "generalChat", "analyzeReview",
    "summarizeText", "unknown"
]

### פעולות מערכת התשלומים

בחלק זה ממומשות הפעולות המרכזיות של מערכת התשלומים.

המערכת מאפשרת יצירת משתמשים, בדיקת יתרות, העברת כספים,
יצירת בקשות תשלום, אישור או דחיית בקשות וצפייה בהיסטוריית העסקאות.

בכל פעולה מתבצעות בדיקות תקינות בהתאם לכללים העסקיים,
והתוצאות נרשמות ביומן המערכת לצורך מעקב וביקורת.

In [9]:
# יצירת משתמש חדש וארנק דיגיטלי
# הפונקציה בודקת את תקינות הנתונים,
# יוצרת משתמש חדש ושומרת את הפעולה ביומן המערכת
def create_user(name: str, phone_number: str, initial_balance: float) -> AgentResult:
    
    # כלל עסקי: לא ניתן ליצור משתמש עם יתרה שלילית
    if initial_balance < 0:
        print_result(f"Cannot create user with negative balance: {initial_balance}₪", "error")
        return AgentResult("PaymentSystem", {"error": "Negative initial balance"}, 0.0)
    
    # יצירת מזהה ייחודי
    user_id = f"user_{len(users_db) + 1}"
    
    # יצירת משתמש וארנק
    users_db[user_id] = User(user_id, name, phone_number)
    wallets_db[user_id] = Wallet(user_id, initial_balance)
    
    # רישום ביומן
    audit_log.append(f"[{datetime.now().isoformat()}] Created user {name} (ID: {user_id}) with balance {initial_balance}₪")
    
    print_result(f"User '{name}' created! ID: {user_id} | Balance: {initial_balance}₪", "success")
    return AgentResult("PaymentSystem", {"user_id": user_id, "name": name, "balance": initial_balance}, 1.0)

In [10]:
# בדיקת יתרה של משתמש
# הפונקציה מאתרת את המשתמש ומחזירה את היתרה הנוכחית בארנק

def get_balance(user_id: str) -> AgentResult:
    
    if user_id not in users_db:
        print_result(f"User '{user_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "User not found"}, 0.0)
    
    balance = wallets_db[user_id].balance
    name    = users_db[user_id].name
    
    print_result(f"{name}'s balance: {balance}₪", "info")
    return AgentResult("PaymentSystem", {"user_id": user_id, "balance": balance}, 1.0)

In [11]:
# הצגת היסטוריית העסקאות של משתמש
# הפונקציה מחזירה את כל העסקאות שבהן המשתמש היה שולח או מקבל
def get_transactions(user_id: str) -> AgentResult:
    
    if user_id not in users_db:
        print_result(f"User '{user_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "User not found"}, 0.0)
    
    # מסננים רק עסקאות של המשתמש הזה
    user_transactions = [
        t for t in transactions_db
        if t.sender_id == user_id or t.receiver_id == user_id
    ]
    
    if not user_transactions:
        print_result(f"No transactions found for {users_db[user_id].name}", "info")
        return AgentResult("PaymentSystem", {"transactions": []}, 1.0)
    
    # הדפסה יפה של כל עסקה
    name = users_db[user_id].name
    print_result(f"Transaction history for {name}:", "info")
    for t in user_transactions:
        sender   = users_db[t.sender_id].name
        receiver = users_db[t.receiver_id].name
        print_result(f"  {sender} → {receiver} | {t.amount}₪ | risk: {t.risk_score} | {t.timestamp[:10]}", "info")
    
    return AgentResult("PaymentSystem", {"transactions": [str(t) for t in user_transactions]}, 1.0)

In [12]:
# העברת כסף בין שני משתמשים
# לפני ביצוע ההעברה מתבצעות כל הבדיקות העסקיות הנדרשות
# לאחר מכן היתרות מתעדכנות והעסקה נשמרת במערכת
def transfer_money(sender_id: str, receiver_id: str, amount: float) -> AgentResult:
    
    # כלל 1: סכום חייב להיות חיובי
    if amount <= 0:
        print_result(f"Invalid amount: {amount}₪. Must be positive!", "error")
        return AgentResult("PaymentSystem", {"error": "Invalid amount"}, 0.0)
    
    # כלל 2: שני המשתמשים חייבים להתקיים
    if sender_id not in users_db:
        print_result(f"Sender '{sender_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "Sender not found"}, 0.0)
    
    if receiver_id not in users_db:
        print_result(f"Receiver '{receiver_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "Receiver not found"}, 0.0)
    
    # כלל 3: אי אפשר להעביר לעצמך
    if sender_id == receiver_id:
        print_result("Cannot transfer money to yourself!", "error")
        return AgentResult("PaymentSystem", {"error": "Self transfer"}, 0.0)
    
    # כלל 4: יתרה מספקת
    sender_balance = wallets_db[sender_id].balance
    if sender_balance < amount:
        print_result(f"Insufficient funds! Balance: {sender_balance}₪, Required: {amount}₪", "error")
        return AgentResult("PaymentSystem", {"error": "Insufficient funds"}, 0.0)
    
    # כל הבדיקות עברו - מבצעים את ההעברה!
    wallets_db[sender_id].balance   -= amount
    wallets_db[receiver_id].balance += amount
    
    # שומרים את העסקה
    transaction = Transaction(
        sender_id   = sender_id,
        receiver_id = receiver_id,
        amount      = amount
    )
    transactions_db.append(transaction)
    
    # רישום ביומן
    sender_name   = users_db[sender_id].name
    receiver_name = users_db[receiver_id].name
    audit_log.append(f"[{datetime.now().isoformat()}] {sender_name} → {receiver_name}: {amount}₪")
    
    print_result(f"Transfer successful! {sender_name} → {receiver_name} | {amount}₪", "success")
    print_result(f"{sender_name}'s new balance: {wallets_db[sender_id].balance}₪", "info")
    
    return AgentResult("PaymentSystem", {
        "transaction": transaction,
        "sender_new_balance": wallets_db[sender_id].balance
    }, 1.0)

In [13]:
# יצירת בקשת תשלום
# משתמש אחד יכול לבקש ממשתמש אחר להעביר סכום כסף מסוים
def request_payment(requester_id: str, payer_id: str, amount: float) -> AgentResult:
    
    if requester_id not in users_db or payer_id not in users_db:
        print_result("One or both users not found!", "error")
        return AgentResult("PaymentSystem", {"error": "User not found"}, 0.0)
    
    if amount <= 0:
        print_result(f"Invalid amount: {amount}₪", "error")
        return AgentResult("PaymentSystem", {"error": "Invalid amount"}, 0.0)
    
    # יצירת מזהה ייחודי לבקשה
    request_counter[0] += 1
    request_id = f"req_{request_counter[0]}"
    
    payment_requests_db[request_id] = PaymentRequest(
        request_id   = request_id,
        requester_id = requester_id,
        payer_id     = payer_id,
        amount       = amount
    )
    
    requester_name = users_db[requester_id].name
    payer_name     = users_db[payer_id].name
    audit_log.append(f"[{datetime.now().isoformat()}] Payment request {request_id}: {requester_name} requests {amount}₪ from {payer_name}")
    
    print_result(f"Payment request created! {requester_name} requests {amount}₪ from {payer_name} | ID: {request_id}", "info")
    return AgentResult("PaymentSystem", {"request_id": request_id, "amount": amount}, 1.0)


In [14]:
# אישור בקשת תשלום
# לאחר האישור מתבצעת העברת הכסף והבקשה מסומנת כמאושרת
def approve_payment_request(request_id: str) -> AgentResult:
    
    if request_id not in payment_requests_db:
        print_result(f"Request '{request_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "Request not found"}, 0.0)
    
    req = payment_requests_db[request_id]
    
    # כלל עסקי: לא ניתן לאשר בקשה שכבר טופלה
    if req.status != "pending":
        print_result(f"Request '{request_id}' already {req.status}!", "error")
        return AgentResult("PaymentSystem", {"error": f"Already {req.status}"}, 0.0)
    
    # מבצעים את ההעברה
    result = transfer_money(req.payer_id, req.requester_id, req.amount)
    
    if "error" in result.output:
        req.status = "failed"
        return result
    
    req.status = "approved"
    audit_log.append(f"[{datetime.now().isoformat()}] Request {request_id} approved")
    
    print_result(f"Payment request {request_id} approved! ✓", "success")
    return AgentResult("PaymentSystem", {"request_id": request_id, "status": "approved"}, 1.0)


In [15]:
# דחיית בקשת תשלום
# הבקשה מסומנת כדחויה ללא ביצוע העברת כספים
def reject_payment_request(request_id: str) -> AgentResult:
    
    if request_id not in payment_requests_db:
        print_result(f"Request '{request_id}' not found!", "error")
        return AgentResult("PaymentSystem", {"error": "Request not found"}, 0.0)
    
    req = payment_requests_db[request_id]
    
    if req.status != "pending":
        print_result(f"Request '{request_id}' already {req.status}!", "error")
        return AgentResult("PaymentSystem", {"error": f"Already {req.status}"}, 0.0)
    
    req.status = "rejected"
    audit_log.append(f"[{datetime.now().isoformat()}] Request {request_id} rejected")
    
    print_result(f"Payment request {request_id} rejected!", "warning")
    return AgentResult("PaymentSystem", {"request_id": request_id, "status": "rejected"}, 1.0)

### סוכן שפה מקומי

סוכן זה אחראי על משימות כלליות כגון שיחה חופשית,
סיכום טקסטים ומתן תשובות כלליות.

הסוכן משתמש במודל שפה מקומי באמצעות Ollama.
במקרה שהמודל אינו זמין, מופעל מנגנון גיבוי המחזיר תשובה בסיסית,
וכך המערכת ממשיכה לפעול ללא תלות בשירות חיצוני.

In [16]:
def call_ollama(prompt : str) -> Optional[str]:
    if requests  is None :
        return None
    try:
        payload = {"model": OLLAMA_MODEL,"prompt": prompt,"stream" : False}
        response = request.post(OLLAMA_URL,json=payload, timeout=20)
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except Exceptation :
        return None

def local_llm_agent(prompt :str) -> AgentResult:
    answer = call_ollama(prompt)
    if not answer:
        answer = "Local fallback answer: I received the request and will answer briefly "
    return AgentResult ("LocalLLMAgent", answer,0.80 )

### סוכן ניתוב (Router Agent)

סוכן הניתוב אחראי לזהות את כוונת המשתמש מתוך ההודעה שהתקבלה.

לאחר זיהוי הכוונה, הסוכן מחזיר את סוג הפעולה המבוקשת
יחד עם רמת ביטחון בזיהוי.

בהמשך המידע מועבר לרכיב התזמור,
אשר מחליט איזה סוכן או כלי יטפל בבקשה.

In [17]:
def router_agent(user_text: str) -> AgentResult:
    text = user_text.lower()
    
    # כוונות מערכת התשלומים
    if any(w in text for w in ["create user", "new user", "register", "add user"]):
        intent, conf = "createUser", 0.92

    elif any(w in text for w in ["balance", "how much", "funds"]):
        intent, conf = "checkBalance", 0.91

    elif any(w in text for w in ["transfer", "send money", "send "]):
        intent, conf = "transferMoney", 0.93

    elif any(w in text for w in ["request payment", "ask for money", "charge"]):
        intent, conf = "requestPayment", 0.90

    elif any(w in text for w in ["approve", "confirm", "accept"]):
        intent, conf = "approvePayment", 0.91

    elif any(w in text for w in ["reject", "decline", "deny"]):
        intent, conf = "rejectPayment", 0.91

    elif any(w in text for w in ["transactions", "history", "show transfers"]):
        intent, conf = "showTransactions", 0.90

    elif any(w in text for w in ["fraud", "suspicious", "unusual"]):
        intent, conf = "fraudCheck", 0.92

    elif any(w in text for w in ["security", "review", "audit"]):
        intent, conf = "securityReview", 0.89

    elif any(w in text for w in ["explain", "why", "what happened", "last action"]):
        intent, conf = "explainLastAction", 0.88

    # כוונות מתרגיל 4 - נשמרות!
    elif any(w in text for w in ["analyze", "sentiment"]):
        intent, conf = "analyzeReview", 0.93

    elif any(w in text for w in ["summarize", "summary"]):
        intent, conf = "summarizeText", 0.90

    elif len(text.strip()) < 4:
        intent, conf = "unknown", 0.40

    else:
        intent, conf = "generalChat", 0.82

    return AgentResult("RouterAgent", {"intent": intent, "confidence": conf}, conf)

In [18]:
alice = create_user("Alice", "050-1111111", 1000)
bob   = create_user("Bob",   "050-2222222", 500)

### סוכן ניתוח רגשות

סוכן זה אחראי לזהות את הרגש המרכזי בטקסט שהתקבל.

המערכת מנסה תחילה להשתמש במודל למידת מכונה מוכן של Transformers.
אם המודל אינו זמין, מופעל מנגנון גיבוי המבוסס על כללים פשוטים ומילות מפתח.

גישה זו מבטיחה שהמערכת תוכל להמשיך לפעול גם ללא מודל חיצוני.

In [19]:
# SentimentAgent
try:
    from transformers import pipeline
    sentiment_pipeline = pipeline("sentiment-analysis")
except Exception:
    sentiment_pipeline = None

# מנגנון גיבוי לניתוח רגשות
# מזהה רגשות חיוביים ושליליים באמצעות מילות מפתח
def rule_based_sentiment(text: str) -> Dict[str, Any]:
    positive = ["good", "great", "amazing", "excellent", "love"]
    negative = ["bad", "terrible", "awful", "hate", "poor"]
    lower = text.lower()
    if any(w in lower for w in positive):
        return {"sentiment": "POSITIVE", "confidence": 0.85}
    if any(w in lower for w in negative):
        return {"sentiment": "NEGATIVE", "confidence": 0.85}
    return {"sentiment": "NEUTRAL", "confidence": 0.60}


# סוכן ניתוח רגשות
# משתמש במודל למידת מכונה כאשר הוא זמין
# אחרת משתמש במנגנון גיבוי מבוסס חוקים
def sentiment_agent(text: str) -> AgentResult:
    if sentiment_pipeline:
        result = sentiment_pipeline(text)[0]
        output = {"sentiment": result["label"], "confidence": round(float(result["score"]), 3)}
    else:
        output = rule_based_sentiment(text)
    return AgentResult("SentimentAnalysisAgent", output, output["confidence"])

### בחירת סוכן לביצוע המשימה

לאחר שזוהתה כוונת המשתמש, המערכת צריכה לבחור
איזה סוכן או כלי יבצע את הפעולה.

רכיב זה אחראי למפות בין הכוונה שזוהתה
לבין הסוכן המתאים ביותר לביצוע המשימה.

בנוסף, אם רמת הביטחון נמוכה מדי,
המערכת מפעילה מנגנון גיבוי.

In [20]:
# רשימת הכלים והסוכנים הזמינים במערכת
# כל כלי אחראי על סוג משימות שונה
TOOLS = {
    "local_llm": "general chat and simple summarization",
    "sentiment_analyzer": "sentiment analysis for reviews",
    "cloud_fallback": "fallback when confidence is low"
}

# בחירת הסוכן המתאים בהתאם לכוונה שזוהתה
# ולרמת הביטחון שהוחזרה על ידי סוכן הניתוב
def select_tool(intent: str, confidence: float) -> str:
    # במקרה של רמת ביטחון נמוכה מופעל מנגנון גיבוי
    if confidence < 0.80:
        return "cloud_fallback"

    # כוונות הקשורות למערכת התשלומים
    payment_intents = [
        "createUser", "checkBalance", "transferMoney",
        "requestPayment", "approvePayment", "rejectPayment",
        "showTransactions"
    ]
    
    if intent in payment_intents:
        return "payment_system"
    elif intent == "fraudCheck":
        return "fraud_agent"
    elif intent == "securityReview":
        return "security_agent"
    elif intent == "explainLastAction":
        return "explanation_agent"
    elif intent == "analyzeReview":
        return "sentiment_analyzer"
    elif intent == "generalChat":
        return "local_llm"
    else:
        return "cloud_fallback"
      

### זיכרון המערכת

המערכת שומרת מידע על פעולות קודמות לצורך שמירת הקשר בין בקשות שונות.

הזיכרון מאפשר לסוכנים לזכור פעולות אחרונות, עסקאות שבוצעו,
בקשות תשלום ותוצאות קודמות.

כך ניתן לספק הסברים, לבצע מעקב אחר פעולות המשתמש
ולשמור רצף עבודה לאורך זמן.

In [21]:
# זיכרון משותף לכל הסוכנים במערכת
# משמש לשמירת מידע מפעולות קודמות לצורך מעקב והסברים
agent_memory = {
    # מתרגיל 4 - נשמר
    "last_intent":       None,
    "last_tool":         None,
    "last_user_message": None,
    "last_result":       None,

    # חדש - זיכרון עסקי
    "last_action":          None,
    "last_user_id":         None,
    "last_transaction":     None,
    "last_payment_request": None,
    "recent_actions":       []
}


 

In [22]:
# עדכון זיכרון המערכת לאחר ביצוע פעולה
# הפונקציה שומרת מידע על הבקשה, התוצאה והפעולה שבוצעה
def update_memory(intent: str, tool: str, user_message: str, 
                  result: AgentResult, action: str = None,
                  user_id: str = None, transaction: Transaction = None,
                  payment_request: PaymentRequest = None):
    
    # עדכון שדות מתרגיל 4
    agent_memory["last_intent"]       = intent
    agent_memory["last_tool"]         = tool
    agent_memory["last_user_message"] = user_message
    agent_memory["last_result"]       = result.output

    # עדכון שדות עסקיים חדשים
    if action:
        agent_memory["last_action"] = action
        # שומרים רק 5 פעולות אחרונות
        agent_memory["recent_actions"].append(action)
        if len(agent_memory["recent_actions"]) > 5:
            agent_memory["recent_actions"].pop(0)

    if user_id:
        agent_memory["last_user_id"] = user_id

    if transaction:
        agent_memory["last_transaction"] = transaction

    if payment_request:
        agent_memory["last_payment_request"] = payment_request

### סוכן ביקורת

סוכן הביקורת אחראי להעריך את איכות התוצאות המוחזרות על ידי הסוכנים השונים במערכת.

הסוכן בודק את איכות התשובה, את רמת הביטחון ואת מבנה הפלט,
ומחליט האם ניתן לקבל את התוצאה או שיש צורך במנגנון גיבוי.

גישה זו מוסיפה שכבת בקרה נוספת ומשפרת את אמינות המערכת.

In [23]:
# סוכן ביקורת
# מעריך את איכות התוצאה שהוחזרה על ידי סוכן אחר
# ומחליט האם ניתן לקבל את התוצאה או שיש להפעיל מנגנון גיבוי

def critic_agent(result: AgentResult) -> AgentResult:
    
    text  = str(result.output)
    score = 3  # ציון ברירת מחדל
    
    # בדיקות שמורידות ציון
    if "error" in result.output if isinstance(result.output, dict) else False:
        score = 1  # תוצאה עם שגיאה - גרועה מאוד
        
    elif len(text.strip()) < 20:
        score = 2  # תשובה קצרה מדי
        
    elif result.confidence < 0.5:
        score = 2  # ביטחון נמוך מדי
    
    # בדיקות שמעלות ציון
    elif result.confidence >= 0.9 and len(text) > 30:
        score = 5  # ביטחון גבוה + תשובה מפורטת
        
    elif "POSITIVE" in text or "NEGATIVE" in text or "amount" in text.lower():
        score = 4  # תשובה מובנית וברורה
        
    elif result.confidence >= 0.8:
        score = 3  # סביר
    
    # החלטה: האם צריך fallback?
    needs_fallback = score < 3
    
    if needs_fallback:
        print_result(f"CriticAgent: score {score}/5 — fallback needed! ⚠️", "warning")
    else:
        print_result(f"CriticAgent: score {score}/5 — result accepted ✓", "success")
    
    return AgentResult(
        agent_name = "CriticAgent",
        output     = {"quality_score": score, "needs_fallback": needs_fallback},
        confidence = score / 5,
        metadata   = {"original_agent": result.agent_name}
    )

### סוכן הסברים

סוכן זה אחראי להסביר למשתמש מה קרה בפעולה האחרונה שבוצעה במערכת.

הסוכן משתמש במידע השמור בזיכרון המערכת,
ומסוגל להסביר העברות כספים, יצירת משתמשים,
אישור או דחיית בקשות תשלום ובדיקות הונאה.

בנוסף, הסוכן מציג היסטוריה קצרה של פעולות אחרונות
כדי לספק הקשר רחב יותר למשתמש.

In [24]:
# סוכן הסברים
# משתמש בזיכרון המערכת כדי להסביר למשתמש
# מה הייתה הפעולה האחרונה שבוצעה ומה הייתה התוצאהסוכן הסבר - מסביר למשתמש מה קרה בפעולה האחרונה
def explanation_agent(user_text: str = None) -> AgentResult:
    
    last_intent      = agent_memory["last_intent"]
    last_action      = agent_memory["last_action"]
    last_result      = agent_memory["last_result"]
    last_transaction = agent_memory["last_transaction"]
    recent_actions   = agent_memory["recent_actions"]
    
    # אם אין מידע בזיכרון
    if not last_action and not last_intent:
        print_result("No previous action found in memory!", "warning")
        return AgentResult("ExplanationAgent", {"explanation": "No previous action to explain"}, 0.5)
    
    # בניית הסבר לפי סוג הפעולה האחרונה
    if last_intent == "transferMoney" and last_transaction:
        sender   = users_db[last_transaction.sender_id].name
        receiver = users_db[last_transaction.receiver_id].name
        explanation = (
            f"Last action: Money transfer.\n"
            f"{sender} sent {last_transaction.amount}₪ to {receiver}.\n"
            f"Risk score: {last_transaction.risk_score}.\n"
            f"Status: {last_transaction.status}."
        )
        
    elif last_intent == "transferMoney" and last_result:
        if "error" in last_result:
            explanation = f"Last transfer failed. Reason: {last_result['error']}."
        else:
            explanation = f"Last action: {last_action}."
            
    elif last_intent == "createUser":
        explanation = f"Last action: New user was created. Details: {last_action}."
        
    elif last_intent == "approvePayment":
        explanation = f"Last action: Payment request was approved. Details: {last_action}."
        
    elif last_intent == "rejectPayment":
        explanation = f"Last action: Payment request was rejected. Details: {last_action}."
        
    elif last_intent == "fraudCheck":
        explanation = f"Last action: Fraud check was performed. Result: {last_result}."
        
    else:
        explanation = f"Last action: {last_action or last_intent}."
    
    # הוספת היסטוריה קצרה
    if recent_actions:
        history = " → ".join(recent_actions[-3:])  # 3 פעולות אחרונות
        explanation += f"\n\nRecent history: {history}"
    
    print_result(f"Explanation: {explanation}", "info")
    return AgentResult("ExplanationAgent", {"explanation": explanation}, 0.9)

In [25]:
# בדיקת סוכן הסברים

# קודם נבצע העברה כדי שיהיה מה להסביר
alice_to_bob = transfer_money("user_1", "user_2", 300)
update_memory(
    intent       = "transferMoney",
    tool         = "payment_system",
    user_message = "Transfer 300₪ to Bob",
    result       = alice_to_bob,
    action       = "Transfer 300₪ from Alice to Bob",
    user_id      = "user_1",
    transaction  = transactions_db[-1]  # העסקה האחרונה
)

print("\n=== הסבר על הפעולה האחרונה ===")
explanation_agent("Explain the last transaction")


=== הסבר על הפעולה האחרונה ===


AgentResult(agent_name='ExplanationAgent', output={'explanation': 'Last action: Money transfer.\nAlice sent 300₪ to Bob.\nRisk score: 0.0.\nStatus: completed.\n\nRecent history: Transfer 300₪ from Alice to Bob'}, confidence=0.9, metadata=None)

### סוכן אבטחה

סוכן האבטחה אחראי לבצע ביקורת פנימית על מערכת התשלומים.

הסוכן בודק תקינות נתונים, שלמות עסקאות,
קיום יומן פעולות, מצב הארנקים ובקשות התשלום.

מטרתו לזהות בעיות אפשריות ולוודא שהמערכת פועלת
בהתאם לכללים העסקיים והאבטחתיים שהוגדרו.

In [26]:
 #SecurityAgent סוכן אבטחה - בודק שהמערכת עובדת בצורה בטוחה
def security_agent() -> AgentResult:
    
    issues   = []  # בעיות שנמצאו
    passed   = []  # בדיקות שעברו
    
    # בדיקה 1: האם יש audit_log?
    if len(audit_log) > 0:
        passed.append("✓ Audit log is active")
    else:
        issues.append("✗ Audit log is empty!")
    
    # בדיקה 2: האם כל העסקאות חיוביות?
    negative_transfers = [t for t in transactions_db if t.amount <= 0]
    if not negative_transfers:
        passed.append("✓ No negative transfers found")
    else:
        issues.append(f"✗ Found {len(negative_transfers)} negative transfers!")
    
    # בדיקה 3: האם אף משתמש ביתרה שלילית?
    negative_balances = [
        uid for uid, w in wallets_db.items() 
        if w.balance < 0
    ]
    if not negative_balances:
        passed.append("✓ No negative balances found")
    else:
        issues.append(f"✗ Users with negative balance: {negative_balances}")
    
    # בדיקה 4: האם בקשות תשלום כפולות?
    approved = [
        r for r in payment_requests_db.values() 
        if r.status == "approved"
    ]
    duplicate_check = len(approved) == len(set(r.request_id for r in approved))
    if duplicate_check:
        passed.append("✓ No duplicate approvals found")
    else:
        issues.append("✗ Duplicate payment approvals detected!")
    
    # בדיקה 5: האם יש משתמשים ללא ארנק?
    users_without_wallet = [
        uid for uid in users_db 
        if uid not in wallets_db
    ]
    if not users_without_wallet:
        passed.append("✓ All users have wallets")
    else:
        issues.append(f"✗ Users without wallet: {users_without_wallet}")
    
    # תוצאה סופית
    score      = len(passed) / (len(passed) + len(issues))
    all_passed = len(issues) == 0
    
    print_result(f"Security Review — {len(passed)}/{len(passed)+len(issues)} checks passed", 
                 "success" if all_passed else "warning")
    
    for p in passed:
        print_result(p, "success")
    for i in issues:
        print_result(i, "error")
    
    return AgentResult(
        agent_name = "SecurityAgent",
        output     = {"passed": passed, "issues": issues, "all_passed": all_passed},
        confidence = score
    )

In [27]:
# # הרצת בדיקת אבטחה לדוגמה
print("=== ביקורת אבטחה ===")
security_agent()

=== ביקורת אבטחה ===


AgentResult(agent_name='SecurityAgent', output={'passed': ['✓ Audit log is active', '✓ No negative transfers found', '✓ No negative balances found', '✓ No duplicate approvals found', '✓ All users have wallets'], 'issues': [], 'all_passed': True}, confidence=1.0, metadata=None)

### סוכן זיהוי הונאות

סוכן זה אחראי לזהות עסקאות חשודות במערכת התשלומים.

הסוכן בוחן מספר מאפיינים של העסקה,
כגון גודל הסכום, תדירות ההעברות והתנהגות המשתמש,
ומחשב ציון סיכון עבור כל עסקה.

מטרת הסוכן היא לזהות דפוסים חריגים שעשויים להעיד
על פעילות לא תקינה או ניסיון הונאה.

In [28]:
# FraudDetectionAgent  סוכן זיהוי הונאות
def fraud_detection_agent(transaction: Transaction) -> AgentResult:
    
    risk_score = 0.0
    reasons    = []
    
    sender_balance = wallets_db[transaction.sender_id].balance
    total_balance  = sender_balance + transaction.amount  # היתרה לפני ההעברה
    
    # קריטריון 1: סכום גבוה מ-80% מהיתרה
    if total_balance > 0:
        percentage = transaction.amount / total_balance
        if percentage >= 0.8:
            risk_score += 0.4
            reasons.append(f"⚠️ Amount is {percentage*100:.0f}% of total balance")
    
    # קריטריון 2: סכום גבוה מ-1000₪
    if transaction.amount > 1000:
        risk_score += 0.3
        reasons.append(f"⚠️ Large amount: {transaction.amount}₪ (over 1000₪)")
    
    # קריטריון 3: יותר מ-3 עסקאות של אותו שולח
    sender_transactions = [
        t for t in transactions_db 
        if t.sender_id == transaction.sender_id
    ]
    if len(sender_transactions) > 3:
        risk_score += 0.2
        reasons.append(f"⚠️ High frequency: {len(sender_transactions)} transactions")
    
    # קריטריון 4: העברה לאותו נמען מספר פעמים
    same_receiver = [
        t for t in transactions_db
        if t.sender_id == transaction.sender_id 
        and t.receiver_id == transaction.receiver_id
    ]
    if len(same_receiver) > 2:
        risk_score += 0.1
        reasons.append(f"⚠️ Repeated transfers to same receiver: {len(same_receiver)} times")
    
    # עדכון ציון הסיכון בעסקה עצמה
    transaction.risk_score = round(min(risk_score, 1.0), 2)
    
    # החלטה סופית
    if risk_score >= 0.7:
        status = "HIGH RISK 🔴"
        status_type = "error"
    elif risk_score >= 0.4:
        status = "MEDIUM RISK 🟡"
        status_type = "warning"
    else:
        status = "LOW RISK 🟢"
        status_type = "success"
    
    sender_name   = users_db[transaction.sender_id].name
    receiver_name = users_db[transaction.receiver_id].name
    
    print_result(
        f"Fraud Check: {sender_name} → {receiver_name} | "
        f"{transaction.amount}₪ | {status} (score: {transaction.risk_score})",
        status_type
    )
    
    for reason in reasons:
        print_result(reason, "warning")
    
    if not reasons:
        print_result("✓ No suspicious patterns detected", "success")
    
    return AgentResult(
        agent_name = "FraudDetectionAgent",
        output     = {
            "risk_score": transaction.risk_score,
            "reasons":    reasons,
            "status":     status
        },
        confidence = 1.0 - transaction.risk_score
    )

In [29]:
# בדיקות FraudDetectionAgent

print("=== עסקה רגילה ===")
t1 = transfer_money("user_1", "user_2", 50)
fraud_detection_agent(transactions_db[-1])

print("\n=== עסקה גדולה ===")
t2 = transfer_money("user_1", "user_2", 1500)
fraud_detection_agent(transactions_db[-1])

print("\n=== עסקה חשודה - 90% מהיתרה ===")
balance = wallets_db["user_2"].balance
t3 = transfer_money("user_2", "user_1", balance * 0.9)
fraud_detection_agent(transactions_db[-1])

=== עסקה רגילה ===



=== עסקה גדולה ===



=== עסקה חשודה - 90% מהיתרה ===


AgentResult(agent_name='FraudDetectionAgent', output={'risk_score': 0.4, 'reasons': ['⚠️ Amount is 90% of total balance'], 'status': 'MEDIUM RISK 🟡'}, confidence=0.6, metadata=None)

 ### סוכן רפלקציה

סוכן הרפלקציה מופעל כאשר פעולה במערכת נכשלת.

מטרת הסוכן היא לנתח את השגיאה שהתקבלה,
להסביר את הסיבה לכשל ולהציע למשתמש דרך לתקן את הבעיה.

כך המערכת אינה מסתפקת בהצגת הודעת שגיאה בלבד,
אלא מספקת הכוונה מעשית להמשך הפעולה.

In [30]:
#ReflectionAgent סוכן רפלקציה - מקבל שגיאה ומציע תיקון למשתמש
def reflection_agent(failed_result: AgentResult, user_text: str) -> AgentResult:
    
    error = None
    suggestion = None
    
    # מחלצים את השגיאה מהתוצאה
    if isinstance(failed_result.output, dict):
        error = failed_result.output.get("error", None)
    
    if not error:
        print_result("No error found — reflection not needed", "info")
        return AgentResult("ReflectionAgent", {"suggestion": "No error to reflect on"}, 1.0)
    
    # מציעים תיקון לפי סוג השגיאה
    if error == "Insufficient funds":
        balance = None
        # מנסים למצוא את היתרה של המשתמש מהזיכרון
        last_user = agent_memory["last_user_id"]
        if last_user and last_user in wallets_db:
            balance = wallets_db[last_user].balance
        suggestion = (
            f"Transfer failed due to insufficient funds. "
            f"{'Current balance: ' + str(balance) + '₪. ' if balance else ''}"
            f"Try transferring a smaller amount."
        )

    elif error == "Self transfer":
        suggestion = (
            "You cannot transfer money to yourself. "
            "Please provide a different receiver."
        )

    elif error == "User not found":
        suggestion = (
            "One of the users was not found. "
            "Check the user ID and try again. "
            "Use 'create user' to add a new user."
        )

    elif error == "Invalid amount":
        suggestion = (
            "The amount must be a positive number. "
            "Please enter a valid amount greater than 0."
        )

    elif error == "Negative initial balance":
        suggestion = (
            "Cannot create a user with negative balance. "
            "Please enter an initial balance of 0 or more."
        )

    elif "Already" in str(error):
        suggestion = (
            f"This payment request was already processed ({error}). "
            "You cannot approve or reject it again."
        )
    elif error == "not found" in str(error).lower():  # ← добавили вторую проверку
        suggestion = (
            "One of the users was not found. "
            "Check the user ID and try again. "
            "Use 'create user' to add a new user."
        )

    else:
        suggestion = (
            f"An error occurred: {error}. "
            "Please check your input and try again."
        )
    
    
    print_result(f"💡 Suggestion: {suggestion}", "info")
    
    return AgentResult(
        agent_name = "ReflectionAgent",
        output     = {
            "original_error": error,
            "suggestion":     suggestion,
            "original_input": user_text
        },
        confidence = 0.85
    )

In [31]:
# בדיקות ReflectionAgent
print("=== יתרה לא מספקת ===")
failed = transfer_money("user_1", "user_2", 99999)
reflection_agent(failed, "Transfer 99999₪ to Bob")

print("\n=== העברה לעצמך ===")
failed = transfer_money("user_1", "user_1", 100)
reflection_agent(failed, "Transfer 100₪ to myself")

print("\n=== משתמש לא קיים ===")
failed = transfer_money("user_1", "user_99", 100)
reflection_agent(failed, "Transfer 100₪ to user_99")

print("\n=== סכום לא חוקי ===")
failed = transfer_money("user_1", "user_2", -50)
reflection_agent(failed, "Transfer -50₪ to Bob")

=== יתרה לא מספקת ===



=== העברה לעצמך ===



=== משתמש לא קיים ===



=== סכום לא חוקי ===


AgentResult(agent_name='ReflectionAgent', output={'original_error': 'Invalid amount', 'suggestion': 'The amount must be a positive number. Please enter a valid amount greater than 0.', 'original_input': 'Transfer -50₪ to Bob'}, confidence=0.85, metadata=None)

### סוכן התזמור

סוכן התזמור הוא הרכיב המרכזי במערכת.

תפקידו לקבל את בקשת המשתמש, לזהות את הכוונה,
לבחור את הסוכן המתאים, להפעיל אותו,
לבצע בדיקות נוספות במידת הצורך ולעדכן את זיכרון המערכת.

הסוכן משמש כגורם המתאם בין כל הסוכנים
ומנהל את תהליך קבלת ההחלטות מקצה לקצה.

In [32]:
# OrchestratorAgent
# סוכן גיבוי
# מופעל כאשר לא ניתן לזהות את הבקשה
# או כאשר רמת הביטחון נמוכה מדי
def cloud_fallback_agent(user_text: str) -> AgentResult:
    answer = "Fallback answer: the request is unclear or confidence is too low."
    return AgentResult("CloudFallbackAgent", answer, 0.70)

# סוכן התזמור הראשי
# אחראי על ניהול זרימת העבודה בין כל הסוכנים במערכת
def orchestrator_agent(user_text: str,
                       user_id: str = None,
                       target_id: str = None,
                       amount: float = None,
                       request_id: str = None) -> AgentResult:
    
    print_result(f"🎯 Orchestrator: '{user_text}'", "info")
    
    # שלב 1: ניתוב
    route      = router_agent(user_text)
    intent     = route.output["intent"]
    confidence = route.output["confidence"]
    tool       = select_tool(intent, confidence)
    
    print_result(f"→ Intent: {intent} | Tool: {tool} | Confidence: {confidence}", "info")
    
    result = None
    
    # שלב 2: הפעלת הכלי המתאים
    
    if tool == "payment_system":
        
        if intent == "createUser" and user_id and amount is not None:
            result = create_user(user_id, "000-0000000", amount)
            
        elif intent == "checkBalance" and user_id:
            result = get_balance(user_id)
            
        elif intent == "transferMoney" and user_id and target_id and amount:
            result = transfer_money(user_id, target_id, amount)
            
            # אחרי העברה - תמיד בודקים הונאות!
            if result.confidence == 1.0 and transactions_db:
                print_result("→ Running fraud check...", "info")
                fraud_result = fraud_detection_agent(transactions_db[-1])
                
                # אם סיכון גבוה - מפעילים דיון
                if fraud_result.output["risk_score"] >= 0.4:
                    print_result("→ Risk detected! Starting debate...", "warning")
                    multi_agent_debate(transactions_db[-1])
                    
            # אם הייתה שגיאה - מפעילים ReflectionAgent
            if result.confidence == 0.0:
                print_result("→ Transfer failed, reflecting...", "warning")
                reflection_agent(result, user_text)
                
        elif intent == "requestPayment" and user_id and target_id and amount:
            result = request_payment(user_id, target_id, amount)
            
        elif intent == "approvePayment" and request_id:
            result = approve_payment_request(request_id)
            
        elif intent == "rejectPayment" and request_id:
            result = reject_payment_request(request_id)
            
        elif intent == "showTransactions" and user_id:
            result = get_transactions(user_id)
            
        else:
            result = cloud_fallback_agent(user_text)
    
    elif tool == "fraud_agent":
        if transactions_db:
            result = fraud_detection_agent(transactions_db[-1])
        else:
            result = cloud_fallback_agent(user_text)
            
    elif tool == "security_agent":
        result = security_agent()
        
    elif tool == "explanation_agent":
        result = explanation_agent(user_text)
        
    elif tool == "sentiment_analyzer":
        result = sentiment_agent(user_text)
        
    elif tool == "local_llm":
        result = local_llm_agent(user_text)
        
    else:
        result = cloud_fallback_agent(user_text)
    
   # שלב 3: CriticAgent בודק את התוצאה
    critic_result = critic_agent(result)

    if critic_result.output["quality_score"] < 3:
        reflection = reflection_agent(result, user_text)

        result.metadata = {
            "critic_score": critic_result.output["quality_score"],
            "reflection_used": True
        }
        
    # שלב 4: עדכון זיכרון
    update_memory(
        intent       = intent,
        tool         = tool,
        user_message = user_text,
        result       = result,
        action       = f"{intent} by {user_id or 'unknown'}",
        user_id      = user_id,
        transaction  = transactions_db[-1] if transactions_db else None
    )
    
    return result

In [33]:
#  בדיקות OrchestratorAgent

print("=== יצירת משתמש ===")
orchestrator_agent("Create a new user", user_id="Dana", amount=2000)

print("\n=== בדיקת יתרה ===")
orchestrator_agent("What is my balance?", user_id="user_1")

print("\n=== העברה חשודה ===")
orchestrator_agent("Transfer money", user_id="user_3", target_id="user_1", amount=900)

print("\n=== הסבר על הפעולה האחרונה ===")
orchestrator_agent("Explain what happened")

print("\n=== ביקורת אבטחה ===")
orchestrator_agent("Security review please")

=== יצירת משתמש ===



=== בדיקת יתרה ===



=== העברה חשודה ===



=== הסבר על הפעולה האחרונה ===



=== ביקורת אבטחה ===


AgentResult(agent_name='SecurityAgent', output={'passed': ['✓ Audit log is active', '✓ No negative transfers found', '✓ No negative balances found', '✓ No duplicate approvals found', '✓ All users have wallets'], 'issues': [], 'all_passed': True}, confidence=1.0, metadata=None)

### דיון בין סוכנים

רכיב זה מדגים שיתוף פעולה בין מספר סוכנים לצורך קבלת החלטה.

כאשר מזוהה עסקה חשודה, מספר סוכנים בוחנים את העסקה
מנקודות מבט שונות ומציגים את המלצתם.

לאחר מכן מתקבלת החלטה סופית על בסיס חוות הדעת
של כלל הסוכנים המשתתפים בדיון.

In [34]:
# דיון בין סוכנים
# מספר סוכנים בוחנים את אותה העסקה
# ומקבלים החלטה משותפת לגבי אישור או חסימה
def multi_agent_debate(transaction: Transaction) -> AgentResult:
    
    sender_name   = users_db[transaction.sender_id].name
    receiver_name = users_db[transaction.receiver_id].name
    
    print_result(f"⚔️ Multi-Agent Debate started for: "
                 f"{sender_name} → {receiver_name} | {transaction.amount}₪", "warning")
    print_result("─" * 50, "info")
    
    # ===== סוכן הונאות מציג את הטיעונים שלו =====
    fraud_result  = fraud_detection_agent(transaction)
    fraud_score   = fraud_result.output["risk_score"]
    fraud_reasons = fraud_result.output["reasons"]
    
    fraud_vote = "BLOCK" if fraud_score >= 0.4 else "APPROVE"
    print_result(f"🔍 FraudAgent vote: {fraud_vote} "
                 f"(risk score: {fraud_score})", 
                 "error" if fraud_vote == "BLOCK" else "success")
    
    # ===== סוכן האבטחה מציג את הטיעונים שלו =====
    security_result = security_agent()
    security_passed = security_result.output["all_passed"]
    security_issues = security_result.output["issues"]
    
    security_vote = "APPROVE" if security_passed else "BLOCK"
    print_result(f"🛡️ SecurityAgent vote: {security_vote} "
                 f"({'all checks passed' if security_passed else str(len(security_issues)) + ' issues found'})",
                 "success" if security_vote == "APPROVE" else "error")
    
    print_result("─" * 50, "info")
    
    # ===== החלטה סופית =====
    # שני הסוכנים צריכים להסכים כדי לאשר
    if fraud_vote == "BLOCK" and security_vote == "BLOCK":
        final_decision = "BLOCKED"
        reason = "Both agents flagged this transaction"
        decision_type = "error"
        
    elif fraud_vote == "BLOCK" and security_vote == "APPROVE":
        # במקרה של חילוקי דעות - ציון הסיכון מכריע
        if fraud_score >= 0.7:
            final_decision = "BLOCKED"
            reason = f"FraudAgent override: risk score {fraud_score} is too high"
            decision_type = "error"
        else:
            final_decision = "APPROVED WITH WARNING"
            reason = f"SecurityAgent approved but monitor closely (risk: {fraud_score})"
            decision_type = "warning"
            
    elif fraud_vote == "APPROVE" and security_vote == "BLOCK":
        final_decision = "BLOCKED"
        reason = "Security issues must be resolved first"
        decision_type = "error"
        
    else:
        final_decision = "APPROVED"
        reason = "Both agents approved the transaction"
        decision_type = "success"
    
    print_result(f"⚖️ Final Decision: {final_decision}", decision_type)
    print_result(f"📋 Reason: {reason}", decision_type)
    
    return AgentResult(
        agent_name = "MultiAgentDebate",
        output     = {
            "final_decision": final_decision,
            "reason":         reason,
            "fraud_vote":     fraud_vote,
            "security_vote":  security_vote,
            "fraud_score":    fraud_score
        },
        confidence = 1.0 - fraud_score
    )

In [35]:
# בדיקות Multi-Agent Debate

print("=== עסקה חשודה מאוד ===")
# קודם יוצרים משתמש עשיר
rich = create_user("Rich", "050-9999999", 10000)
big_transfer = transfer_money("user_3", "user_1", 9000)
multi_agent_debate(transactions_db[-1])

print("\n\n=== עסקה רגילה ===")
normal = transfer_money("user_1", "user_2", 50)
multi_agent_debate(transactions_db[-1])

=== עסקה חשודה מאוד ===




=== עסקה רגילה ===


AgentResult(agent_name='MultiAgentDebate', output={'final_decision': 'APPROVED', 'reason': 'Both agents approved the transaction', 'fraud_vote': 'APPROVE', 'security_vote': 'APPROVE', 'fraud_score': 0.1}, confidence=0.9, metadata=None)

### שמירה וטעינה של נתוני המערכת

מערכת התשלומים שומרת את הנתונים לקובץ

JSON

ומאפשרת לטעון אותם מחדש בעת הצורך.

כך ניתן לשמור משתמשים, ארנקים, עסקאות,
בקשות תשלום ויומן פעולות גם לאחר סיום הריצה.

רכיב זה מדמה שכבת אחסון פשוטה ללא שימוש במסד נתונים חיצוני.

In [36]:

# קובץ JSON המשמש לשמירת נתוני המערכת

JSON_FILE = "payment_system_data.json"

def save_to_json() -> AgentResult:
    
    data = {
        "users": {
            uid: {
                "user_id":      u.user_id,
                "name":         u.name,
                "phone_number": u.phone_number
            } for uid, u in users_db.items()
        },
        "wallets": {
            uid: {
                "user_id": w.user_id,
                "balance": w.balance
            } for uid, w in wallets_db.items()
        },
        "transactions": [
            {
                "sender_id":   t.sender_id,
                "receiver_id": t.receiver_id,
                "amount":      t.amount,
                "timestamp":   t.timestamp,
                "status":      t.status,
                "risk_score":  t.risk_score
            } for t in transactions_db
        ],
        "payment_requests": {
            rid: {
                "request_id":   r.request_id,
                "requester_id": r.requester_id,
                "payer_id":     r.payer_id,
                "amount":       r.amount,
                "status":       r.status,
                "created_at":   r.created_at
            } for rid, r in payment_requests_db.items()
        },
        "audit_log": audit_log
    }
    
    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print_result(f"💾 Data saved to '{JSON_FILE}' — "
                 f"{len(users_db)} users, "
                 f"{len(transactions_db)} transactions", "success")
    
    return AgentResult("JSONStorage", {"file": JSON_FILE, "saved": True}, 1.0)


In [37]:
#JSON טעינת נתוני המערכת מקובץ  
# ושחזור כל האובייקטים לזיכרון

def load_from_json() -> AgentResult:
    
    if not Path(JSON_FILE).exists():
        print_result(f"File '{JSON_FILE}' not found!", "error")
        return AgentResult("JSONStorage", {"error": "File not found"}, 0.0)
    
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # טוענים חזרה לזיכרון
    users_db.clear()
    wallets_db.clear()
    transactions_db.clear()
    payment_requests_db.clear()
    audit_log.clear()
    
    for uid, u in data["users"].items():
        users_db[uid] = User(u["user_id"], u["name"], u["phone_number"])
    
    for uid, w in data["wallets"].items():
        wallets_db[uid] = Wallet(w["user_id"], w["balance"])
    
    for t in data["transactions"]:
        transaction = Transaction(
            sender_id   = t["sender_id"],
            receiver_id = t["receiver_id"],
            amount      = t["amount"],
            timestamp   = t["timestamp"],
            status      = t["status"],
            risk_score  = t["risk_score"]
        )
        transactions_db.append(transaction)
    
    for rid, r in data["payment_requests"].items():
        payment_requests_db[rid] = PaymentRequest(
            request_id   = r["request_id"],
            requester_id = r["requester_id"],
            payer_id     = r["payer_id"],
            amount       = r["amount"],
            status       = r["status"],
            created_at   = r["created_at"]
        )
    
    audit_log.extend(data["audit_log"])
    
    print_result(f"📂 Data loaded from '{JSON_FILE}' — "
                 f"{len(users_db)} users, "
                 f"{len(transactions_db)} transactions", "success")
    
    return AgentResult("JSONStorage", {"file": JSON_FILE, "loaded": True}, 1.0)

In [38]:
# בדיקת JSON
print("=== שמירה ===")
save_to_json()

print("\n=== מחיקת הנתונים מהזיכרון ===")
users_db.clear()
wallets_db.clear()
transactions_db.clear()
print_result(f"Memory cleared — users: {len(users_db)}, transactions: {len(transactions_db)}", "warning")

print("\n=== טעינה חזרה ===")
load_from_json()
print_result(f"After load — users: {len(users_db)}, transactions: {len(transactions_db)}", "info")

print("\n=== בדיקת יתרות לאחר טעינה ===")
for uid, user in users_db.items():
    get_balance(uid)

=== שמירה ===



=== מחיקת הנתונים מהזיכרון ===



=== טעינה חזרה ===



=== בדיקת יתרות לאחר טעינה ===


### בדיקות מערכת

חלק זה מבצע סדרת בדיקות אוטומטיות על מערכת התשלומים.

מטרת הבדיקות היא לוודא שכל הרכיבים פועלים כמצופה
ושכל הכללים העסקיים נאכפים בצורה תקינה.

הבדיקות כוללות יצירת משתמשים, העברות כספים,
טיפול בשגיאות, בקשות תשלום ובדיקת מקרי קצה.

In [39]:
#  בדיקות חובה
# מאפסים את המערכת לפני הבדיקות
users_db.clear()
wallets_db.clear()
transactions_db.clear()
payment_requests_db.clear()
audit_log.clear()
request_counter[0] = 0
agent_memory["recent_actions"] = []

print("=" * 60)
print("         FINAL TESTS - מערכת תשלומים")
print("=" * 60)

passed_tests = 0
total_tests  = 8

# ─────────────────────────────────────────
# בדיקה 1: יצירת שני משתמשים ובדיקת יתרות
# ─────────────────────────────────────────
print("\n📋 TEST 1: Create two users and check balances")
alice = create_user("Alice", "050-1111111", 1000)
bob   = create_user("Bob",   "050-2222222", 500)

bal_alice = wallets_db["user_1"].balance
bal_bob   = wallets_db["user_2"].balance

assert alice.output["user_id"] == "user_1", "FAIL: Alice ID"
assert bob.output["user_id"]   == "user_2", "FAIL: Bob ID"
assert bal_alice == 1000, "FAIL: Alice balance"
assert bal_bob   == 500,  "FAIL: Bob balance"

passed_tests += 1
print_result(f"TEST 1 PASSED ✓ — Alice: {bal_alice}₪, Bob: {bal_bob}₪", "success")

# ─────────────────────────────────────────
# בדיקה 2: העברת כסף תקינה
# ─────────────────────────────────────────
print("\n📋 TEST 2: Valid money transfer")
result = transfer_money("user_1", "user_2", 200)
assert result.confidence == 1.0,              "FAIL: Transfer confidence"
assert wallets_db["user_1"].balance == 800,   "FAIL: Alice new balance"
assert wallets_db["user_2"].balance == 700,   "FAIL: Bob new balance"
passed_tests += 1
print_result("TEST 2 PASSED ✓ — Transfer 200 from Alice to Bob", "success")

# ─────────────────────────────────────────
# בדיקה 3: סכום שלילי
# ─────────────────────────────────────────
print("\n📋 TEST 3: Negative amount")
result = transfer_money("user_1", "user_2", -50)
assert "error" in result.output,                   "FAIL: Should return error"
assert result.output["error"] == "Invalid amount", "FAIL: Wrong error"
passed_tests += 1
print_result("TEST 3 PASSED ✓ — Negative amount rejected", "success")

# ─────────────────────────────────────────
# בדיקה 4: יתרה לא מספקת
# ─────────────────────────────────────────
print("\n📋 TEST 4: Insufficient funds")
result = transfer_money("user_2", "user_1", 9999)
assert "error" in result.output,                        "FAIL: Should return error"
assert result.output["error"] == "Insufficient funds",  "FAIL: Wrong error"
assert wallets_db["user_2"].balance == 700,             "FAIL: Balance should not change"
passed_tests += 1
print_result("TEST 4 PASSED ✓ — Insufficient funds rejected", "success")

# ─────────────────────────────────────────
# בדיקה 5: משתמש לא קיים
# ─────────────────────────────────────────
print("\n📋 TEST 5: Non-existent user")
result = transfer_money("user_1", "user_99", 100)
assert "error" in result.output,                      "FAIL: Should return error"
assert "not found" in result.output["error"].lower(), "FAIL: Wrong error"
passed_tests += 1
print_result("TEST 5 PASSED ✓ — Non-existent user rejected", "success")

# ─────────────────────────────────────────
# בדיקה 6: העברה לעצמך
# ─────────────────────────────────────────
print("\n📋 TEST 6: Self transfer")
result = transfer_money("user_1", "user_1", 100)
assert "error" in result.output,                  "FAIL: Should return error"
assert result.output["error"] == "Self transfer", "FAIL: Wrong error"
passed_tests += 1
print_result("TEST 6 PASSED ✓ — Self transfer rejected", "success")

# ─────────────────────────────────────────
# בדיקה 7: בקשת תשלום ואישור
# ─────────────────────────────────────────
print("\n📋 TEST 7: Payment request and approval")
req    = request_payment("user_2", "user_1", 150)
req_id = req.output["request_id"]
assert payment_requests_db[req_id].status == "pending",  "FAIL: Should be pending"
approve_payment_request(req_id)
assert payment_requests_db[req_id].status == "approved", "FAIL: Should be approved"
assert wallets_db["user_1"].balance == 650, "FAIL: Alice balance"
assert wallets_db["user_2"].balance == 850, "FAIL: Bob balance"
passed_tests += 1
print_result(f"TEST 7 PASSED ✓ — Request {req_id} approved", "success")

# ─────────────────────────────────────────
# בדיקה 8: אישור כפול
# ─────────────────────────────────────────
print("\n📋 TEST 8: Double approval")
result = approve_payment_request(req_id)
assert "error" in result.output,            "FAIL: Should return error"
assert "Already" in result.output["error"], "FAIL: Wrong error"
passed_tests += 1
print_result("TEST 8 PASSED ✓ — Double approval rejected", "success")

# תוצאה סופית
print("\n" + "=" * 60)
if passed_tests == total_tests:
    print_result(f"🏆 ALL {total_tests}/{total_tests} TESTS PASSED!", "success")
else:
    print_result(f"⚠️ {passed_tests}/{total_tests} tests passed", "warning")
print("=" * 60)

         FINAL TESTS - מערכת תשלומים

📋 TEST 1: Create two users and check balances



📋 TEST 2: Valid money transfer



📋 TEST 3: Negative amount



📋 TEST 4: Insufficient funds



📋 TEST 5: Non-existent user



📋 TEST 6: Self transfer



📋 TEST 7: Payment request and approval



📋 TEST 8: Double approval


### לוח בקרה

לוח הבקרה מספק תמונת מצב עדכנית של מערכת התשלומים.

הוא מציג נתונים מרכזיים כגון מספר משתמשים,
כמות העסקאות, סכומי כסף, רמות סיכון,
בקשות תשלום ופעולות אחרונות שבוצעו במערכת.

המטרה היא לאפשר ניטור מהיר ונוח של מצב המערכת.

In [105]:
def create_bank_app():
    
    # ─── CSS ───
    display(HTML("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&display=swap');
    .widget-vbox, .widget-hbox {
    background-color: transparent !important;
}
.jp-OutputArea-output {
    background: #050520 !important;
    border-radius: 0 0 20px 20px;
    padding: 20px;
}

    .widget-text input, .widget-float-text input {
    background: #ffffff0d !important;
    border: 1px solid #4ecca322 !important;
    border-radius: 10px !important;
    color: white !important;            /* ← уже есть */
    font-family: 'Outfit', sans-serif !important;
}

 
.widget-text input::placeholder,
.widget-float-text input::placeholder {
    color: #ffffff55 !important;
}

 
.widget-float-text input {
    color: white !important;
}
    .widget-text input:focus, .widget-float-text input:focus {
        border-color: #4ecca3 !important;
        background: #4ecca308 !important;
    }
    .widget-button button {
        font-family: 'Outfit', sans-serif !important;
        font-weight: 600 !important;
        border-radius: 10px !important;
        border: none !important;
        font-size: 13px !important;
        transition: all 0.2s !important;
    }
    .widget-button button.mod-success { background: #4ecca3 !important; color: #0d0d1a !important; }
    .widget-button button.mod-primary { background: #3498db !important; color: white !important; }
    .widget-button button.mod-warning { background: #f39c12 !important; color: #0d0d1a !important; }
    .widget-button button.mod-danger  { background: #e74c3c !important; color: white !important; }
    </style>
    """))

    output = widgets.Output()

    def section(title, *rows):
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="
                background:#ffffff06;
                border:1px solid #4ecca318;
                border-radius:14px;
                padding:16px 18px 6px 18px;
                margin-bottom:4px;
            ">
                <div style="
                    font-size:11px;
                    font-weight:600;
                    color:#4ecca3;
                    letter-spacing:1.5px;
                    text-transform:uppercase;
                    font-family:'Outfit',sans-serif;
                    margin-bottom:10px;
                ">{title}</div>
            </div>"""),
            widgets.VBox(list(rows), layout=widgets.Layout(
                padding="0 0 10px 0",
                border="1px solid #4ecca318",
                border_radius="0 0 14px 14px",
                background_color="#ffffff06",
                margin="0 0 14px 0"
            ))
        ])

    def inp(placeholder, width="170px"):
        return widgets.Text(
            placeholder=placeholder,
            layout=widgets.Layout(width=width)
        )

    def num(value=100, width="120px"):
        return widgets.FloatText(
            value=value,
            layout=widgets.Layout(width=width)
        )

    def btn(label, style="success", width="130px"):
        return widgets.Button(
            description=label,
            button_style=style,
            layout=widgets.Layout(width=width, height="36px")
        )

    # ─── שדות ───
    f_name    = inp("Alice", "180px")
    f_phone   = inp("050-0000000", "160px")
    f_balance = num(1000)
    b_create  = btn("➕ Create", "success")

    f_from    = inp("user_1")
    f_to      = inp("user_2")
    f_amount  = num(100)
    b_transfer = btn("💸 Send", "primary")

    f_uid     = inp("user_1", "300px")
    b_balance = btn("💰 Check", "warning", "140px")

    f_rf      = inp("user_1")
    f_rt      = inp("user_2")
    f_ra      = num(50)
    b_request = btn("📤 Request", "warning")

    f_rid     = inp("req_1", "250px")
    b_approve = btn("✅ Approve", "success")
    b_reject  = btn("❌ Reject", "danger")

    b_dash    = btn("📊 Dashboard", "", "150px")
    b_sec     = btn("🛡️ Security", "", "150px")
    b_save    = btn("💾 Save JSON", "", "150px")
    b_clear   = btn("🗑️ Clear", "warning", "130px")

    # ─── לחיצות ───
    def on_create(b):
        with output:
            create_user(f_name.value, f_phone.value, f_balance.value)

    def on_transfer(b):
        with output:
            result = transfer_money(f_from.value, f_to.value, f_amount.value)
            if result.confidence == 0.0:
                reflection_agent(result, "transfer")
            elif transactions_db:
                fraud = fraud_detection_agent(transactions_db[-1])
                if fraud.output["risk_score"] >= 0.4:
                    multi_agent_debate(transactions_db[-1])

    def on_balance(b):
        with output:
            get_balance(f_uid.value)

    def on_request(b):
        with output:
            request_payment(f_rf.value, f_rt.value, f_ra.value)

    def on_approve(b):
        with output:
            approve_payment_request(f_rid.value)

    def on_reject(b):
        with output:
            reject_payment_request(f_rid.value)

    def on_dashboard(b):
        with output:
            show_dashboard()

    def on_security(b):
        with output:
            security_agent()

    def on_save(b):
        with output:
            save_to_json()

    def on_clear(b):
        output.clear_output()

    b_create.on_click(on_create)
    b_transfer.on_click(on_transfer)
    b_balance.on_click(on_balance)
    b_request.on_click(on_request)
    b_approve.on_click(on_approve)
    b_reject.on_click(on_reject)
    b_dash.on_click(on_dashboard)
    b_sec.on_click(on_security)
    b_save.on_click(on_save)
    b_clear.on_click(on_clear)

    # ─── ממשק ───
    app = widgets.VBox([
        widgets.HTML("""
        <div style="
            background:linear-gradient(135deg,#0d0d1a,#1a1a2e);
            border-radius:20px 20px 0 0;
            padding:24px;
            text-align:center;
            border:1px solid #4ecca322;
            border-bottom:none;
        ">
            <div style="font-size:26px;font-weight:700;color:#4ecca3;
                        letter-spacing:3px;font-family:'Outfit',sans-serif;">
                🏦 AGENTIC BANK
            </div>
            <div style="font-size:11px;color:#4ecca366;letter-spacing:2px;
                        margin-top:6px;font-family:'Outfit',sans-serif;">
                POWERED BY MULTI-AGENT SYSTEM
            </div>
        </div>
        """),

        widgets.VBox([
            # Create User
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>👤 CREATE USER</div>"),
            widgets.HBox([f_name, f_phone, f_balance, b_create]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # Transfer
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>💸 TRANSFER MONEY</div>"),
            widgets.HBox([f_from, f_to, f_amount, b_transfer]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # Balance
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>💰 CHECK BALANCE</div>"),
            widgets.HBox([f_uid, b_balance]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # Payment Request
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>📤 PAYMENT REQUEST</div>"),
            widgets.HBox([f_rf, f_rt, f_ra, b_request]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # Approve/Reject
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>✅ APPROVE / REJECT</div>"),
            widgets.HBox([f_rid, b_approve, b_reject]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # System
            widgets.HTML("<div style='font-size:11px;color:#4ecca3;letter-spacing:1.5px;font-family:Outfit,sans-serif;font-weight:600;padding:4px 0 8px 0;'>⚙️ SYSTEM</div>"),
            widgets.HBox([b_dash, b_sec, b_save, b_clear]),

            widgets.HTML("<div style='border-top:1px solid #4ecca311;margin:14px 0;'></div>"),

            # Output
            widgets.HTML("<div style='font-size:10px;color:#ffffff33;letter-spacing:1px;font-family:Outfit,sans-serif;'>OUTPUT ↓</div>"),
            output

        ], layout=widgets.Layout(
            padding="20px 24px 24px 24px",
            background_color="#fffff",
            border="1px solid #4ecca322",
            border_radius="0 0 20px 20px"
        ))
    ])

    display(app)

In [106]:
create_bank_app()

In [107]:
#Dashboard - תצוגת מצב המערכת
def show_dashboard():
    
    # חישובים
    total_money      = sum(w.balance for w in wallets_db.values())
    total_sent       = sum(t.amount for t in transactions_db)
    high_risk        = [t for t in transactions_db if t.risk_score >= 0.7]
    medium_risk      = [t for t in transactions_db if 0.4 <= t.risk_score < 0.7]
    pending_requests = [r for r in payment_requests_db.values() if r.status == "pending"]
    approved_requests = [r for r in payment_requests_db.values() if r.status == "approved"]
    
    # כותרת
    display(HTML(f"""
    <div style="
        background: linear-gradient(135deg, #1a1a2e, #16213e);
        border-radius: 12px;
        padding: 20px;
        margin: 10px 0;
        font-family: Arial;
        color: white;
    ">
        <h2 style="text-align:center; color:#e94560; margin:0 0 20px 0;">
            💳 PAYMENT SYSTEM DASHBOARD
        </h2>
        
        <!-- שורה 1: סטטיסטיקות כלליות -->
        <div style="display:flex; gap:10px; margin-bottom:10px;">
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#e94560;">
                    {len(users_db)}
                </div>
                <div style="color:#aaa; font-size:12px;">👥 USERS</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#4ecca3;">
                    {total_money:,.0f}₪
                </div>
                <div style="color:#aaa; font-size:12px;">💰 TOTAL BALANCE</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#4ecca3;">
                    {len(transactions_db)}
                </div>
                <div style="color:#aaa; font-size:12px;">📤 TRANSACTIONS</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#f5a623;">
                    {total_sent:,.0f}₪
                </div>
                <div style="color:#aaa; font-size:12px;">💸 TOTAL SENT</div>
            </div>
        </div>
        
        <!-- שורה 2: סיכון והתראות -->
        <div style="display:flex; gap:10px; margin-bottom:10px;">
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#e74c3c;">
                    {len(high_risk)}
                </div>
                <div style="color:#aaa; font-size:12px;">🔴 HIGH RISK</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#f39c12;">
                    {len(medium_risk)}
                </div>
                <div style="color:#aaa; font-size:12px;">🟡 MEDIUM RISK</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#f5a623;">
                    {len(pending_requests)}
                </div>
                <div style="color:#aaa; font-size:12px;">⏳ PENDING REQUESTS</div>
            </div>
            <div style="flex:1; background:#0f3460; border-radius:8px; padding:15px; text-align:center;">
                <div style="font-size:28px; font-weight:bold; color:#2ecc71;">
                    {len(approved_requests)}
                </div>
                <div style="color:#aaa; font-size:12px;">✅ APPROVED REQUESTS</div>
            </div>
        </div>
        
        <!-- שורה 3: יתרות משתמשים -->
        <div style="background:#0f3460; border-radius:8px; padding:15px; margin-bottom:10px;">
            <div style="color:#aaa; font-size:12px; margin-bottom:10px;">
                👥 USER BALANCES
            </div>
            {''.join([f"""
            <div style="display:flex; align-items:center; margin:6px 0;">
                <div style="color:white; width:120px;">
                    {users_db[uid].name}
                </div>
                <div style="flex:1; background:#1a1a2e; border-radius:4px; height:20px; overflow:hidden;">
                    <div style="
                        width:{min(wallets_db[uid].balance / max(w.balance for w in wallets_db.values()) * 100, 100):.0f}%;
                        background: linear-gradient(90deg, #4ecca3, #2ecc71);
                        height:100%;
                        border-radius:4px;
                    "></div>
                </div>
                <div style="color:#4ecca3; width:100px; text-align:right;">
                    {wallets_db[uid].balance:,.0f}₪
                </div>
            </div>
            """ for uid in users_db])}
        </div>
        
        <!-- שורה 4: פעולות אחרונות -->
        <div style="background:#0f3460; border-radius:8px; padding:15px;">
            <div style="color:#aaa; font-size:12px; margin-bottom:10px;">
                🕐 RECENT ACTIONS
            </div>
            {''.join([f'<div style="color:#ccc; font-size:12px; padding:3px 0; border-bottom:1px solid #1a1a2e;">→ {action}</div>' 
                      for action in agent_memory["recent_actions"][-5:]]) 
             or '<div style="color:#666;">No recent actions</div>'}
        </div>
        
    </div>
    """))

# הצגת הדשבורד
show_dashboard()